# Gesture Control — YOLO11n Training (Midterm: SEE → RECOGNIZE)

Self-contained Colab pipeline:

1. Download the **HaGRID Sample 30k 384p** dataset from Kaggle (0.86 GB, about 1,750 images per gesture) and take a subject-balanced subset
2. Preprocess: blur filter, near-duplicate removal, YOLO label conversion
3. Subject-aware **train / val / test** split
4. Train **YOLO11n** (COCO-pretrained, fine-tuned)
5. Evaluate: mAP, precision, recall, F1, confusion matrix, image-level accuracy
6. Evaluate on the **unseen self-collected webcam set** (optional upload)
7. Export **ONNX** for the laptop demo (`src/vision/webcam_detect.py`)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run all cells.
Everything is saved to Google Drive in `MyDrive/SnorlaxTheDog_Gestures/`, so a disconnect doesn't lose work.

| id | class (HaGRID) | gesture | planned robot command |
|---|---|---|---|
| 0 | fist | closed fist | STOP |
| 1 | like | thumbs up | FORWARD |
| 2 | palm | open palm | BACKWARD |
| 3 | rock | rock sign (index + little finger) | LEFT |
| 4 | call | call sign (thumb + little finger) | RIGHT |

## 0. Setup

In [ ]:
%pip install -q "ultralytics>=8.3.0" kagglehub onnx onnxruntime
import torch
ALLOW_CPU = False   # CPU training takes hours; only set True if no GPU is available at all
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise SystemExit('No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then run again from the top.')
else:
    print('WARNING: training on CPU - this will be very slow.')

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/SnorlaxTheDog_Gestures')
except Exception as e:
    print('Google Drive not mounted:', e, '-> saving to /content/outputs (download before the session ends!)')
    OUT = Path('/content/outputs')

ON_DRIVE = str(OUT).startswith('/content/drive')
RAW = Path('/content/raw_hagrid')  # selected subset (local disk: fast, rebuilt from Kaggle each session)
DS = Path('/content/gestures')     # split dataset used for training (rebuilt from RAW)
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for p in (OUT, RAW / 'images', RAW / 'labels', FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
print('Outputs ->', OUT)

## 1. Configuration

In [ ]:
CLASSES = ['fist', 'like', 'palm', 'rock', 'call']   # YOLO class ids 0..4 (must match src/config.py)
DISPLAY = {'fist': 'Fist', 'like': 'Thumbs Up', 'palm': 'Open Palm', 'rock': 'Rock Sign', 'call': 'Call Sign'}

KAGGLE_DATASET = 'innominate817/hagrid-sample-30k-384p'   # HaGRID v1 sample, 384p, with annotations

PER_CLASS = 500            # images per class taken from the sample
SEED = 42
BLUR_THRESHOLD = 60.0      # variance of Laplacian; lower = blurry -> rejected
DUP_HASH_DISTANCE = 4      # dHash Hamming distance; <= this = near-duplicate -> rejected
SPLIT = {'train': 0.70, 'val': 0.20, 'test': 0.10}

TRAIN_CFG = dict(
    epochs=60, imgsz=640, batch=16, optimizer='SGD', lr0=0.01, lrf=0.01, momentum=0.937,
    weight_decay=5e-4, warmup_epochs=3, patience=15, seed=SEED, cos_lr=False,
    # augmentation (all 5 gestures are direction-independent, so horizontal flip is safe)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=10.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, close_mosaic=10,
)
CONF_THRESHOLD = 0.50      # same threshold the live demo uses

## 2. Download the dataset and select the subset

Public Kaggle datasets usually download without a login. If this cell fails with a 401/403 error, create an API token at
kaggle.com → Settings → API, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Colab Secrets** (key icon on the left), and re-run.

Annotation format (HaGRID): per-image `bboxes` as normalized `[top-left x, top-left y, w, h]`, `labels` per hand,
and `user_id`. Only images with **exactly one hand of the target gesture** are kept, and they are spread across
as many different subjects as possible.

In [ ]:
import os, json, random
from collections import Counter
import cv2, numpy as np

try:   # optional Kaggle credentials from Colab Secrets
    from google.colab import userdata
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
SRC = Path(kagglehub.dataset_download(KAGGLE_DATASET))
IMG_ROOT = next(SRC.rglob('hagrid_30k'))
ANN_DIR = next(SRC.rglob('ann_train_val'))
print('dataset at', SRC)

ann = {}
for c in CLASSES:
    existing = {p.stem for p in (IMG_ROOT / f'train_val_{c}').glob('*.jpg')}
    full = json.loads((ANN_DIR / f'{c}.json').read_text())
    ann[c] = {k: v for k, v in full.items() if k in existing}
print({c: len(v) for c, v in ann.items()}, 'annotated images per class')

def blur_score(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

def dhash(img, size=8):
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (size + 1, size), interpolation=cv2.INTER_AREA)
    return int(''.join('1' if b else '0' for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

def to_yolo(cid, b):
    x, y, w, h = b
    c = lambda v: min(max(v, 0.0), 1.0)
    return f'{cid} {c(x + w / 2):.6f} {c(y + h / 2):.6f} {c(w):.6f} {c(h):.6f}'

In [ ]:
def pick_candidates(c, rng):
    '''All usable images of class c, ordered round-robin over subjects.'''
    by_user = {}
    for img_id, a in ann[c].items():
        if a['labels'].count(c) == 1:
            by_user.setdefault(a.get('user_id') or img_id, []).append(img_id)
    users = sorted(by_user); rng.shuffle(users)
    for u in users:
        by_user[u].sort(); rng.shuffle(by_user[u])
    order, depth = [], 0
    while any(len(v) > depth for v in by_user.values()):
        order += [by_user[u][depth] for u in users if len(by_user[u]) > depth]
        depth += 1
    return order

# Rebuild the subset from scratch every run (fast: the data is already local).
for d in (RAW / 'images', RAW / 'labels'):
    for f in d.glob('*'):
        f.unlink()
manifest, rejects = [], {}
for c in CLASSES:
    cid = CLASSES.index(c)
    rng = random.Random(f'{SEED}-{c}')
    kept, hashes, rej = 0, [], Counter()
    for img_id in pick_candidates(c, rng):
        if kept >= PER_CLASS:
            break
        img = cv2.imread(str(IMG_ROOT / f'train_val_{c}' / f'{img_id}.jpg'))
        if img is None:
            rej['unreadable'] += 1; continue
        if blur_score(img) < BLUR_THRESHOLD:
            rej['blurry'] += 1; continue
        h = dhash(img)
        if any(bin(h ^ o).count('1') <= DUP_HASH_DISTANCE for o in hashes):
            rej['duplicate'] += 1; continue
        a = ann[c][img_id]
        stem = f'{c}_{img_id}'
        cv2.imwrite(str(RAW / 'images' / f'{stem}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        (RAW / 'labels' / f'{stem}.txt').write_text(
            '\n'.join(to_yolo(cid, b) for b, l in zip(a['bboxes'], a['labels']) if l == c) + '\n')
        manifest.append({'file': f'{stem}.jpg', 'class': c, 'user_id': a.get('user_id') or img_id,
                         'width': img.shape[1], 'height': img.shape[0], 'dhash': str(h)})
        hashes.append(h); kept += 1
    rejects[c] = dict(rej)
    print(f'[{c}] kept {kept}  rejected {dict(rej)}')

(RAW / 'manifest.json').write_text(json.dumps(manifest, indent=1))
json.dump(rejects, open(TAB / 'preprocessing_rejections.json', 'w'), indent=1)
print('total images:', len(manifest))

## 3. Train / validation / test split

Split by **subject (`user_id`)**: all images of one person go to the same split, so the test set contains people
the model never saw during training. Split ratios: 70 / 20 / 10, balanced per class.

In [ ]:
import shutil, pandas as pd, matplotlib.pyplot as plt

rng = random.Random(SEED)
user_split = {}
df = pd.DataFrame(manifest)
df['split'] = ''
for c in CLASSES:
    sub = df[df['class'] == c]
    groups = list(sub.groupby('user_id').groups.items()); rng.shuffle(groups)
    counts = Counter()
    for user, idx in groups:
        if user not in user_split:
            # assign new subject to the split furthest below its target for this class
            user_split[user] = min(SPLIT, key=lambda s: counts[s] / max(len(sub), 1) - SPLIT[s])
        s = user_split[user]
        df.loc[idx, 'split'] = s
        counts[s] += len(idx)

if DS.exists():
    shutil.rmtree(DS)
for _, r in df.iterrows():
    for kind, ext in (('images', '.jpg'), ('labels', '.txt')):
        d = DS / kind / r['split']; d.mkdir(parents=True, exist_ok=True)
        shutil.copy(RAW / kind / (Path(r['file']).stem + ext), d)

(DS / 'data.yaml').write_text(
    f"path: {DS}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")
df.drop(columns=['dhash']).to_csv(TAB / 'dataset_manifest.csv', index=False)

per_class = df.pivot_table(index='class', columns='split', values='file', aggfunc='count').reindex(CLASSES)[['train', 'val', 'test']]
per_class['total'] = per_class.sum(1)
per_class.loc['TOTAL'] = per_class.sum()
per_class.to_csv(TAB / 'split_per_class.csv')
split_tot = df['split'].value_counts().reindex(['train', 'val', 'test'])
split_table = pd.DataFrame({'samples': split_tot, 'percent': (100 * split_tot / split_tot.sum()).round(1)})
split_table.loc['total'] = [split_tot.sum(), 100.0]
split_table.to_csv(TAB / 'split_summary.csv')
print(per_class, '\n\n', split_table)
print('subjects per split:', df.groupby('split')['user_id'].nunique().to_dict())

ax = per_class.drop('TOTAL')[['train', 'val', 'test']].rename(index=DISPLAY).plot.bar(
    stacked=True, figsize=(8, 4), color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_ylabel('images'); ax.set_xlabel(''); ax.set_title('Dataset distribution per class and split')
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(FIG / 'dataset_distribution.png', dpi=200); plt.show()

In [ ]:
# Sample images per class with ground-truth boxes (for the report)
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(14, 3 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    files = df[(df['class'] == c) & (df['split'] == 'train')]['file'].sample(5, random_state=SEED)
    for k, f in enumerate(files):
        img = cv2.imread(str(RAW / 'images' / f)); h, w = img.shape[:2]
        for line in (RAW / 'labels' / (Path(f).stem + '.txt')).read_text().split('\n'):
            if line.strip():
                _, cx, cy, bw, bh = map(float, line.split())
                cv2.rectangle(img, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)),
                              (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (0, 255, 0), 3)
        axes[r, k].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[r, k].axis('off')
        if k == 0:
            axes[r, k].set_title(DISPLAY[c], loc='left', fontsize=13)
plt.tight_layout(); plt.savefig(FIG / 'class_samples.png', dpi=150); plt.show()

## 4. Train YOLO11n

Checkpoints are saved to Drive after every epoch. Re-running this cell:
* after a **disconnect** → resumes from `last.pt`
* after a **finished** run → skips training and reuses the trained model (set `RETRAIN = True` to train again from scratch)

In [ ]:
import torch
from ultralytics import YOLO
RETRAIN = False
RUNS = OUT / 'runs'
WEIGHTS = RUNS / 'train' / 'weights'
last = WEIGHTS / 'last.pt'

state = 'new'
if last.exists() and not RETRAIN:
    ckpt = torch.load(last, map_location='cpu', weights_only=False)
    # Ultralytics removes the optimizer state from last.pt once training has finished
    state = 'finished' if ckpt.get('optimizer') is None else 'interrupted'
    del ckpt
print('training state:', state)

if state == 'interrupted':
    YOLO(str(last)).train(resume=True)
elif state == 'new':
    if RETRAIN and (RUNS / 'train').exists():
        shutil.rmtree(RUNS / 'train')
    YOLO('yolo11n.pt').train(data=str(DS / 'data.yaml'), project=str(RUNS), name='train',
                             exist_ok=True, plots=True, **TRAIN_CFG)   # COCO-pretrained start
BEST = WEIGHTS / 'best.pt'
print('best weights:', BEST)

In [ ]:
# Training curves
from IPython.display import Image, display
res = pd.read_csv(RUNS / 'train' / 'results.csv'); res.columns = res.columns.str.strip()
res.to_csv(TAB / 'training_history.csv', index=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for k in ['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss']:
    ax[0].plot(res['epoch'], res[k], label=k)
ax[0].set_title('Loss'); ax[0].legend(); ax[0].set_xlabel('epoch')
for k in ['metrics/precision(B)', 'metrics/recall(B)']:
    ax[1].plot(res['epoch'], res[k], label=k.split('/')[1])
ax[1].set_title('Validation precision / recall'); ax[1].legend(); ax[1].set_xlabel('epoch')
for k in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
    ax[2].plot(res['epoch'], res[k], label=k.split('/')[1])
ax[2].set_title('Validation mAP'); ax[2].legend(); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(FIG / 'training_curves.png', dpi=200); plt.show()
print('epochs run:', len(res), '| best val mAP50-95:', round(float(res['metrics/mAP50-95(B)'].max()), 4))

## 5. Evaluation on the held-out test set

* **Detection metrics** (Ultralytics `val`, IoU 0.5): precision, recall, mAP@50, mAP@50-95, confusion matrix
* **Image-level recognition**: the highest-confidence detection (conf ≥ 0.50) is the predicted gesture; *none* = no detection.
  This is the accuracy the live demo actually delivers.

In [ ]:
best = YOLO(str(BEST))
m = best.val(data=str(DS / 'data.yaml'), split='test', conf=0.001, iou=0.6, plots=True,
             project=str(RUNS), name='test_eval', exist_ok=True)
p, r = m.box.p, m.box.r
det = pd.DataFrame({'class': [DISPLAY[c] for c in CLASSES], 'precision': p, 'recall': r,
                    'f1': 2 * p * r / np.maximum(p + r, 1e-9), 'mAP50': m.box.ap50, 'mAP50-95': m.box.ap})
det.loc[len(det)] = ['ALL (mean)', m.box.mp, m.box.mr, det['f1'].mean(), m.box.map50, m.box.map]
det = det.round(4); det.to_csv(TAB / 'test_detection_metrics.csv', index=False)
print(det.to_string(index=False))
for f in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    for src in (RUNS / 'test_eval' / f, RUNS / 'test_eval' / f'Box{f}'):   # newer Ultralytics prefixes curves with "Box"
        if src.exists():
            shutil.copy(src, FIG / f'test_{f}')
display(Image(str(RUNS / 'test_eval' / 'confusion_matrix_normalized.png'), width=600))

In [ ]:
LABELS = CLASSES + ['none']

def image_level_eval(paths, gt, tag, conf=CONF_THRESHOLD):
    preds, confs = [], []
    for i in range(0, len(paths), 32):
        for res in best.predict([str(x) for x in paths[i:i + 32]], conf=conf, verbose=False):
            if len(res.boxes):
                k = int(res.boxes.conf.argmax())
                preds.append(CLASSES[int(res.boxes.cls[k])]); confs.append(float(res.boxes.conf[k]))
            else:
                preds.append('none'); confs.append(0.0)
    cm = np.zeros((len(CLASSES), len(LABELS)), int)
    for g, pr in zip(gt, preds):
        cm[CLASSES.index(g), LABELS.index(pr)] += 1
    tp = np.diag(cm[:, :len(CLASSES)])
    prec = tp / np.maximum(cm[:, :len(CLASSES)].sum(0), 1)
    rec = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    acc = tp.sum() / cm.sum()
    table = pd.DataFrame({'class': [DISPLAY[c] for c in CLASSES], 'precision': prec, 'recall': rec, 'f1': f1,
                          'support': cm.sum(1)})
    table.loc[len(table)] = ['macro avg', prec.mean(), rec.mean(), f1.mean(), cm.sum()]
    table = table.round(4); table.to_csv(TAB / f'{tag}_image_level_metrics.csv', index=False)
    pd.DataFrame(cm, index=CLASSES, columns=LABELS).to_csv(TAB / f'{tag}_confusion_matrix.csv')
    print(f'{tag}: accuracy = {acc:.4f} on {cm.sum()} images'); print(table.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 5.5))
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]}\n{norm[i, j]:.0%}', ha='center', va='center',
                    color='white' if norm[i, j] > 0.5 else 'black', fontsize=9)
    ax.set_xticks(range(len(LABELS)), [DISPLAY.get(l, 'None') for l in LABELS], rotation=30)
    ax.set_yticks(range(len(CLASSES)), [DISPLAY[c] for c in CLASSES])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'{tag.replace("_", " ").title()} — image-level confusion matrix (acc {acc:.1%})')
    plt.tight_layout(); plt.savefig(FIG / f'{tag}_image_confusion_matrix.png', dpi=200); plt.show()
    return pd.DataFrame({'path': [str(x) for x in paths], 'gt': gt, 'pred': preds, 'conf': confs}), acc

test_df = df[df['split'] == 'test']
test_paths = [DS / 'images' / 'test' / f for f in test_df['file']]
test_preds, test_acc = image_level_eval(test_paths, list(test_df['class']), 'test')
test_preds.to_csv(TAB / 'test_predictions.csv', index=False)

## 6. Unseen situations — self-collected webcam images (optional)

On the laptop, capture images with `python src/data/capture_webcam.py --person <name> --place <tag>`,
zip the `dataset/webcam` folder as **`webcam.zip`** (it must contain `fist/`, `like/`, `palm/`, `rock/`, `call/`
and optionally `capture_log.csv`), and upload it to `MyDrive/SnorlaxTheDog_Gestures/`.
These images are **never** used for training or validation. The label of each image is the folder it was saved in.

In [ ]:
import zipfile
WEBCAM_ZIP = OUT / 'webcam.zip'
unseen_preds = None
if WEBCAM_ZIP.exists():
    UNSEEN = Path('/content/unseen'); shutil.rmtree(UNSEEN, ignore_errors=True)
    zipfile.ZipFile(WEBCAM_ZIP).extractall(UNSEEN)
    paths, gt = [], []
    for c in CLASSES:
        found = [x for x in UNSEEN.rglob('*') if x.parent.name == c and x.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        paths += found; gt += [c] * len(found)
    print({c: gt.count(c) for c in CLASSES}, 'unseen images')
    unseen_preds, unseen_acc = image_level_eval(paths, gt, 'unseen_webcam')
    unseen_preds.to_csv(TAB / 'unseen_predictions.csv', index=False)
    logs = list(UNSEEN.rglob('capture_log.csv'))
    if logs:   # accuracy per person / place (lighting, background)
        log = pd.read_csv(logs[0]); log['key'] = log['file'].str.replace('\\', '/')
        unseen_preds['key'] = unseen_preds['path'].apply(lambda s: '/'.join(Path(s).parts[-2:]))
        merged = unseen_preds.merge(log, on='key', how='left'); merged['correct'] = merged['gt'] == merged['pred']
        by = merged.groupby(['person', 'place'])['correct'].agg(['mean', 'count']).rename(columns={'mean': 'accuracy'})
        by.round(4).to_csv(TAB / 'unseen_accuracy_by_condition.csv'); print(by.round(3))
else:
    print('No webcam.zip found in', OUT, '- skipping (re-run this cell after uploading it).')

## 7. Sample results (successes and failures)

In [ ]:
def show_examples(preds, title, fname, n=6):
    ok = preds[preds['gt'] == preds['pred']].sample(min(n, (preds['gt'] == preds['pred']).sum()), random_state=SEED)
    bad = preds[preds['gt'] != preds['pred']].head(n)
    rows = [('Correct', ok), ('Incorrect', bad)]
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))
    for r, (name, sub) in enumerate(rows):
        for k in range(n):
            a = axes[r, k]; a.axis('off')
            if k >= len(sub):
                continue
            row = sub.iloc[k]
            res = best.predict(row['path'], conf=CONF_THRESHOLD, verbose=False)[0]
            a.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
            a.set_title(f"{name}\nGT: {DISPLAY[row['gt']]}\nPred: {DISPLAY.get(row['pred'], 'None')} ({row['conf']:.2f})", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG / fname, dpi=150); plt.show()

show_examples(test_preds, 'Test set predictions', 'test_examples.png')
if unseen_preds is not None:
    show_examples(unseen_preds, 'Unseen webcam predictions', 'unseen_examples.png')

## 8. Export ONNX for the laptop demo + package results

In [ ]:
import onnxruntime as ort
onnx_path = Path(best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False))
shutil.copy(onnx_path, OUT / 'best.onnx'); shutil.copy(BEST, OUT / 'best.pt')
sess = ort.InferenceSession(str(OUT / 'best.onnx'))
print('ONNX input', sess.get_inputs()[0].shape, '-> output', sess.get_outputs()[0].shape)

import torch, ultralytics, platform
summary = {
    'classes': CLASSES, 'images_per_class_hagrid': PER_CLASS, 'train_cfg': TRAIN_CFG,
    'split_summary': split_table['samples'].to_dict(),
    'test_detection': det.iloc[-1].to_dict(), 'test_image_accuracy': round(float(test_acc), 4),
    'unseen_image_accuracy': round(float(unseen_acc), 4) if unseen_preds is not None else None,
    'epochs_run': int(len(res)),
    'versions': {'python': platform.python_version(), 'ultralytics': ultralytics.__version__,
                 'torch': torch.__version__, 'opencv': cv2.__version__,
                 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'},
}
json.dump(summary, open(OUT / 'summary.json', 'w'), indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))

# One zip with everything the report needs (weights + figures + tables + Ultralytics plots)
pkg = OUT / 'midterm_results.zip'
with zipfile.ZipFile(pkg, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in [OUT / 'best.onnx', OUT / 'best.pt', OUT / 'summary.json']:
        z.write(f, f.name)
    for d in (FIG, TAB, RUNS / 'train', RUNS / 'test_eval'):
        for f in d.rglob('*'):
            if f.is_file() and f.suffix in ('.png', '.jpg', '.csv', '.json', '.yaml'):
                z.write(f, f.relative_to(OUT))
print('Results package:', pkg, f'({pkg.stat().st_size / 1e6:.1f} MB)')
if not ON_DRIVE:   # no Drive: send the package to the browser instead
    try:
        from google.colab import files
        files.download(str(pkg))
    except Exception as e:
        print('Automatic download failed (', e, ') - download', pkg, 'from the Colab file browser.')